In [1]:
import numpy as np
import pandas as pd
import toolbox
import psutil

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from Lib14.data_properties import HDV_LIG14

import multiprocessing
from multiprocessing.shared_memory import SharedMemory
import concurrent.futures

# Define file location of ".prob" files
filepath_prob = "Datasets/HDV/prob/"

# Get system stats
nb_cores = psutil.cpu_count()
nb_threads = multiprocessing.cpu_count()

KeyboardInterrupt: 

In [2]:
def extract_image(index):
    # Go through all the ".prob" files
    filename = filepath_prob + "SEQUENCE_" + str(index) + ".prob"

    # Extract data from ".prob" file
    image = np.loadtxt(filename, delimiter="\t")

    # Get matrix indexes of upper triangular part with offset of 2
    indexes = np.triu_indices(130, 2)

    # Reshape the image into a compact format (129 x 64)
    image = np.reshape(image[indexes], (129, 64))

    # Return the data
    return image

In [3]:
# Define workers' task
def worker(works):
    jobs = [None] * len(works)
    for i, index in enumerate(works):
        jobs[i] = extract_image(index)
    return list(zip(works, jobs))

In [ ]:
def process(workload):
    # Set the number of threads per process
    nb_workers = nb_threads/nb_cores

    # Divide the workload across workers
    workload_per_thread = np.array_split(workload, nb_workers)   

    # Split further the process into threads
    with concurrent.futures.ThreadPoolExecutor(max_workers=nb_workers) as executor:
        results = list(executor.map(worker, workload_per_thread))

    
    
    # Concatenate the results from each thread
    # results = np.concatenate(results)

    # Return the dataset
    return results

In [ ]:
if __name__ == '__main__':
    # Print system stats
    print("Num CPU Cores Available: ", nb_cores)
    print("Num CPU Threads Available: ", nb_threads)

    # Get the expected output data
    hdv_fit = np.array(HDV_LIG14.hdv_fitness)

    # Get length of the dataset
    len_sequences = HDV_LIG14.seq_amount

    # Initialise dataset
    dataset = np.empty((len_sequences, 129, 64), dtype=np.float32)

    # Divide the workload
    workload_per_core = np.array_split(range(len_sequences), nb_cores)
    
    # Use multiprocessing to populate the dataset in parallel
    with multiprocessing.Pool(processes=nb_cores) as pool:
        results = list(pool.map(process, workload_per_core))

In [ ]:
print(results)